# Mikage Google Execution Lane - Colab Runner

**Purpose:** Job runner for Imagen API + Vertex RAG in Colab environment

**Contract:** Uses `render_job_payload.json` and produces `result_bundle.json`

In [ ]:
# Cell 1: Install dependencies
!pip install google-cloud-aiplatform google-cloud-discoveryengine --quiet

In [ ]:
# Cell 2: Mount Google Drive (for shared storage)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Configuration
import os

# Set these values
PROJECT_ID = "gen-lang-client-0440215253"  # Your GCP project
LOCATION = "us-central1"
DATA_STORE_ID = "mikage-brain_1774647248976"  # Vertex RAG datastore

# Shared root contract
SHARED_ROOT = "/content/drive/MyDrive/mikage_runner"
JOB_INBOX = Path(SHARED_ROOT) / "job_inbox"
OUTPUT_BASE = Path(SHARED_ROOT) / "outputs"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

In [ ]:
# Cell 4: Imports
import json
import base64
import time
from datetime import datetime
from pathlib import Path
from google.cloud import aiplatform
from google.cloud.aiplatform.gapic.schema import predict
from google.cloud import discoveryengine
import google.auth

print("[INIT] Imports complete")

In [ ]:
# Cell 5: RAG Retrieval Function
def call_vertex_rag(query, project_id, location, data_store_id):
    """Call Vertex RAG to retrieve relevant context"""
    try:
        client = discoveryengine.SearchServiceClient()
        serving_config = f"projects/{project_id}/locations/{location}/collections/default_collection/dataStores/{data_store_id}/servingConfigs/default_config"
        
        request = discoveryengine.SearchRequest(
            serving_config=serving_config,
            query=query,
            page_size=5
        )
        
        response = client.search(request)
        
        chunks = []
        for idx, result in enumerate(response.results):
            doc = result.document
            if doc:
                chunks.append({
                    "id": f"chunk_{idx+1}",
                    "content": doc.name or "",
                    "score": result.relevance_score or 0.0,
                    "metadata": {
                        "source": doc.name,
                        "id": doc.id
                    }
                })
        
        return {
            "retriever_mode": "vertex",
            "real_vertex_verified": True,
            "query": query,
            "context": f"Retrieved {len(chunks)} chunks",
            "chunks": chunks
        }
    except Exception as e:
        print(f"[RAG] Error: {e}")
        return {
            "retriever_mode": "vertex",
            "real_vertex_verified": False,
            "query": query,
            "context": "",
            "chunks": [],
            "error": str(e)
        }

In [ ]:
# Cell 6: Imagen API Function
def call_imagen(prompt, negative_prompt="", aspect_ratio="1:1", seed=None, guidance_scale=7.5):
    """Call Imagen 3 API"""
    try:
        aiplatform.init(project=PROJECT_ID, location=LOCATION)
        
        model = aiplatform.ImageGenerationModel.from_pretrained("imagen-3.0-generate-001")
        
        result = model.generate_images(
            prompt=prompt,
            negative_prompt=negative_prompt,
            number_of_images=1,
            aspect_ratio=aspect_ratio,
            seed=seed,
            guidance_scale=guidance_scale
        )
        
        # Extract image data
        images = []
        for img in result.images:
            images.append({
                "bytes_base64": base64.b64encode(img._as_base64_string()).decode(),
                "mime_type": "image/png"
            })
        
        return {
            "predictions": images,
            "success": True
        }
    except Exception as e:
        print(f"[IMAGEN] Error: {e}")
        return {
            "error": True,
            "error_message": str(e),
            "success": False
        }

In [ ]:
# Cell 7: Main Execution Flow
def find_next_job_file():
    JOB_INBOX.mkdir(parents=True, exist_ok=True)
    job_files = sorted(JOB_INBOX.glob("*.json"))
    if not job_files:
        raise FileNotFoundError(f"No job files found in {JOB_INBOX}")
    return job_files[0]

def execute_job():
    """Execute render job end-to-end"""
    started_at = datetime.utcnow().isoformat() + "Z"
    
    # Load job payload
    job_path = find_next_job_file()
    with open(job_path, 'r') as f:
        job = json.load(f)
    
    job_id = job['job_id']
    output_dir = Path(OUTPUT_BASE) / job_id
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"COLAB_PICKED_UP {job_id}")
    
    print(f"[EXEC] Job: {job_id}")
    print(f"[EXEC] Output: {output_dir}")
    
    # Initialize result bundle
    result = {
        "version": "1.0.0",
        "job_id": job_id,
        "status": "FAIL",
        "output_files": [],
        "primary_output": None,
        "render_payload": job,
        "render_response_raw": None,
        "rag_context": None,
        "execution_lane_info": {
            "platform": "colab",
            "runtime": "python",
            "gpu_available": False,
            "execution_timestamp": started_at
        },
        "error": None,
        "timing": {
            "started_at": started_at,
            "rag_completed_at": None,
            "imagen_started_at": None,
            "imagen_completed_at": None,
            "total_duration_ms": 0
        },
        "started_at": started_at,
        "completed_at": None
    }
    
    try:
        # Step 1: RAG (if enabled)
        if job.get('rag_enabled', False):
            print("[EXEC] RAG enabled - retrieving context...")
            rag_query = job.get('rag_query', f"{job.get('shot_type', '')} {job.get('user_idea', '')}")
            result['rag_context'] = call_vertex_rag(rag_query, PROJECT_ID, LOCATION, DATA_STORE_ID)
            result['timing']['rag_completed_at'] = datetime.utcnow().isoformat() + "Z"
            print(f"[EXEC] RAG: {len(result['rag_context']['chunks'])} chunks")
        
        # Step 2: Call Imagen
        print("[EXEC] Calling Imagen API...")
        result['timing']['imagen_started_at'] = datetime.utcnow().isoformat() + "Z"
        
        imagen_result = call_imagen(
            prompt=job['prompt'],
            negative_prompt=job.get('negative_prompt', ''),
            aspect_ratio=job.get('aspect_ratio', '1:1'),
            seed=job.get('seed'),
            guidance_scale=job.get('imagen_config', {}).get('guidance_scale', 7.5)
        )
        
        result['render_response_raw'] = imagen_result
        result['timing']['imagen_completed_at'] = datetime.utcnow().isoformat() + "Z"
        
        # Step 3: Save images
        if imagen_result.get('success') and imagen_result.get('predictions'):
            for idx, pred in enumerate(imagen_result['predictions']):
                img_data = base64.b64decode(pred['bytes_base64'])
                filename = f"{job_id}.png" if idx == 0 else f"{job_id}_{idx+1}.png"
                filepath = output_dir / filename
                
                with open(filepath, 'wb') as f:
                    f.write(img_data)
                
                result['output_files'].append({
                    "path": str(filepath),
                    "type": "image",
                    "mime_type": "image/png",
                    "size_bytes": len(img_data)
                })
                
                if idx == 0:
                    result['primary_output'] = str(filepath)
                
                print(f"[EXEC] Saved: {filepath}")
        else:
            raise Exception("IMAGEN FAILED: No images generated")
        
        # HARD FAIL if no images
        if len(result['output_files']) == 0:
            raise Exception("HARD FAIL: No output images")
        
        result['status'] = "SUCCESS"
        
    except Exception as e:
        print(f"[EXEC] FAILED: {e}")
        result['status'] = "FAIL"
        result['error'] = {
            "code": "EXECUTION_FAILED",
            "message": str(e),
            "stage": "imagen_api",
            "retryable": False
        }
    
    # Finalize
    completed_at = datetime.utcnow().isoformat() + "Z"
    result['completed_at'] = completed_at
    result['timing']['total_duration_ms'] = int(
        (datetime.fromisoformat(completed_at.replace('Z', '+00:00')) - 
         datetime.fromisoformat(started_at.replace('Z', '+00:00'))).total_seconds() * 1000
    )
    
    # Save artifacts
    with open(output_dir / "result.json", 'w') as f:
        json.dump(result, f, indent=2)
    print(f"COLAB_RESULT_WRITTEN {output_dir / 'result.json'}")
    
    with open(output_dir / "render_payload.json", 'w') as f:
        json.dump(job, f, indent=2)
    
    if result['rag_context']:
        with open(output_dir / "rag_context.json", 'w') as f:
            json.dump(result['rag_context'], f, indent=2)
    
    print(f"\n[EXEC] Status: {result['status']}")
    print(f"[EXEC] Images: {len(result['output_files'])}")
    print(f"[EXEC] Duration: {result['timing']['total_duration_ms']}ms")
    
    return result

# Execute
result = execute_job()

In [ ]:
# Cell 8: Verify output
from IPython.display import Image, display

if result['status'] == 'SUCCESS' and result['primary_output']:
    print("[VERIFY] Primary output:")
    display(Image(result['primary_output']))
else:
    print(f"[VERIFY] Job failed: {result.get('error', {}).get('message', 'Unknown error')}")